# Math Journal

A living reference for math concepts encountered while building the robot simulator.  
New concepts are added here as they come up — one entry per topic.

**Primary visual reference:** 3Blue1Brown — Essence of Linear Algebra — https://www.3blue1brown.com/topics/linear-algebra

---

## Contents

1. [Eigenvalues and Eigenvectors](#eigenvalues)
2. [Positive Definite Matrices](#positive-definite)

---

<a id='eigenvalues'></a>
## 1. Eigenvalues and Eigenvectors

**Reference:** 3Blue1Brown, Essence of Linear Algebra, Chapter 14 — https://www.3blue1brown.com/topics/linear-algebra  
**Reference:** Gilbert Strang, *Introduction to Linear Algebra*, Ch. 6

---

### Intuition First

A matrix is a transformation — it takes a vector and moves it somewhere else. Usually that means rotating it *and* stretching it at the same time.

But for most matrices, there exist a few special directions where the vector **does not rotate at all** — it only gets stretched (or compressed, or flipped). These special directions are called **eigenvectors**.

The **eigenvalue** tells you *how much* the vector was stretched in that direction.

> **Analogy:** Imagine pressing a blob of dough flat. Most directions get shifted diagonally. But if you press straight down, that direction only shrinks — it doesn't rotate sideways. That "straight down" direction is an eigenvector. How much it shrinks is the eigenvalue.

---

### The Equation

$$
M \mathbf{v} = \lambda \mathbf{v}
$$

Where:
- $M$ — a square matrix (e.g. the mass matrix)
- $\mathbf{v}$ — the eigenvector (a non-zero vector that doesn't rotate under $M$)
- $\lambda$ — the eigenvalue (a scalar: how much $\mathbf{v}$ was stretched)

Read it as: *"Applying M to v gives the same result as just scaling v by λ."*

---

### What the Eigenvalue Tells You

| Eigenvalue $\lambda$ | Meaning |
|---|---|
| $\lambda > 1$ | The eigenvector gets stretched longer |
| $0 < \lambda < 1$ | The eigenvector gets compressed shorter |
| $\lambda = 1$ | No change — identity in that direction |
| $\lambda = 0$ | The eigenvector collapses to zero — the matrix is **singular** (non-invertible) |
| $\lambda < 0$ | The eigenvector gets flipped and scaled |

An $n \times n$ matrix has $n$ eigenvalues (counting repeats). Each one corresponds to a different eigenvector direction.

---

### How to Compute in NumPy

```python
import numpy as np

M = np.array([[4, 1], [2, 3]])

eigenvalues = np.linalg.eigvals(M)
# Returns an array of eigenvalues, one per row/column

# Check all are positive:
assert np.all(eigenvalues > 0)
```

For symmetric matrices (like the mass matrix), use `np.linalg.eigvalsh` instead — it is faster and guaranteed to return real numbers.

---

### Why It Matters in Robotics

The mass matrix $M(\boldsymbol{\theta})$ must always have **all positive eigenvalues**. This is because:

$$
T = \frac{1}{2} \dot{\boldsymbol{\theta}}^T M \dot{\boldsymbol{\theta}}
$$

Kinetic energy $T$ must always be positive (you cannot have negative kinetic energy). If any eigenvalue were zero or negative, you could find a joint velocity $\dot{\boldsymbol{\theta}}$ that produces zero or negative kinetic energy — physically impossible.

A **zero eigenvalue** in $M$ would also mean $M$ is singular (non-invertible), which means you cannot solve for $\ddot{\boldsymbol{\theta}}$ in the equation of motion — the simulation breaks down. This is related to **singular configurations** of the arm.

---

<a id='positive-definite'></a>
## 2. Positive Definite Matrices

**Reference:** Gilbert Strang, *Introduction to Linear Algebra*, Ch. 6.5

---

### Intuition

A matrix is **positive definite** if, when you use it to "measure" any non-zero vector, you always get a positive number.

Formally: a symmetric matrix $M$ is positive definite if for every non-zero vector $\mathbf{x}$:

$$
\mathbf{x}^T M \mathbf{x} > 0
$$

This is exactly the kinetic energy expression $T = \frac{1}{2} \dot{\boldsymbol{\theta}}^T M \dot{\boldsymbol{\theta}}$ — positive definite means kinetic energy is always positive.

---

### The Eigenvalue Test

For a symmetric matrix, these are all equivalent:

| Test | What it means |
|---|---|
| All eigenvalues $> 0$ | Positive definite |
| All eigenvalues $\geq 0$ | Positive **semi**-definite (allows zero energy — singular) |
| Any eigenvalue $\leq 0$ | Not positive definite — something is wrong |

The eigenvalue test is the easiest to apply in code, which is why it's the standard check for the mass matrix.

---

### Symmetry

The mass matrix is also always **symmetric**: $M = M^T$.

This comes from the structure of the Jacobian formula — $J_v^T J_v$ is always symmetric regardless of what $J_v$ contains. Symmetry is a quick sanity check you can run before the eigenvalue test:

```python
assert np.allclose(M, M.T)
```

---